# Import Libraries

In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

# Check request

In [9]:
home_url = "https://books.toscrape.com/"
try:
    response = requests.get(home_url, timeout=10)
    response.encoding='utf-8'
    response.raise_for_status()
    print("status code:",response.status_code)
except requests.RequestException as error:
    print("Request failed:", error)

# print(response.text[:1000])
# print( "valid"if response.status_code>=200 and response.status_code<300 else "Not Valid"  )

status code: 200


#example on first book

In [10]:
soup= BeautifulSoup(response.text,"html.parser")
print("Page title:", soup.title.get_text(strip=True))
books = soup.find_all("article", class_="product_pod")

#len books for each page
num_books = len(books)
print("\nNumber of books in one page:", num_books)
#First Book
first_book = books[0]
#title
title = first_book.find("h3").find("a")["title"]
print("Title:", title)
#price
price=first_book.find("p",class_="price_color").get_text(strip=True)
print("Price:", price)
#Availability
availability = first_book.find("p", class_="instock availability").get_text(strip=True)
print("Availability:", availability)
#Rating
rating = first_book.find("p", class_="star-rating")["class"][1]
print("Rating:", rating)
#link()
product_link = first_book.find("h3").find("a")["href"]
print("Product Link:", product_link)#there is a little problem we are going to fix it later
# print(f"product link:{home_url+product_link}")

Page title: All products | Books to Scrape - Sandbox

Number of books in one page: 20
Title: A Light in the Attic
Price: £51.77
Availability: In stock
Rating: Three
Product Link: catalogue/a-light-in-the-attic_1000/index.html


#Collect Data "Books" into List

In [11]:
all_books = []
#Page Loops
for page in range(1, 51):

    # Choose Url
    if page == 1:
        url = home_url
    else:
        url = f"{home_url}catalogue/page-{page}.html"
    # Request the current page
    response = requests.get(url)
    response.encoding = 'utf-8'

    # Parse the current page
    soup = BeautifulSoup(response.text, "html.parser")

    # Find books on the current page
    books = soup.find_all("article", class_="product_pod")

#Book Loops
    for book in books:
        #title
        title = book.find("h3").find("a")["title"]

        #price
        price=book.find("p",class_="price_color").get_text(strip=True)

        #Availability
        availability = book.find("p", class_="instock availability").get_text(strip=True)

        #Rating
        rating = book.find("p", class_="star-rating")["class"][1]
        #link
        product_link =urljoin(url,book.find("h3").find("a")["href"])

        all_books.append({
          "title": title,
          "price": price,
          "availability": availability,
          "rating": rating,
          "product_link": product_link
          })
print("\nScraping completed.")
print("Total books:", len(all_books))


Scraping completed.
Total books: 1000


#Save Data as a CSV file

In [12]:
df = pd.DataFrame(all_books)
df.info()#infos about dataset
df.head()#first 5 rows
df.shape#shape of dataset

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         1000 non-null   object
 1   price         1000 non-null   object
 2   availability  1000 non-null   object
 3   rating        1000 non-null   object
 4   product_link  1000 non-null   object
dtypes: object(5)
memory usage: 39.2+ KB


(1000, 5)

In [13]:
print(df.isnull().sum())#Check missing values
print("\nNum of duplicated rows:" ,df.duplicated().sum())#Check duplicated

title           0
price           0
availability    0
rating          0
product_link    0
dtype: int64

Num of duplicated rows: 0


In [15]:
from google.colab import files

# Export using the 'utf-8-sig' encoding to support Microsoft Excel
df.to_csv("books.csv", index=False, encoding="utf-8-sig")

# Download the corrected file
files.download("books_excel_compatible.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>